### Structured Output

Models can be requested to provide their response in a format matching a given schema . This is useful for ensuring the output can be easily parsed nad used in subsequent processing . LangChain supports multilple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.


In [25]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("qwen/qwen3.8-27b", model_provider="groq")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, client=<groq.resources.chat.completions.Completions object at 0x000001B990A4BAD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B990C904D0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [22]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="This is the director of the movie")
    ratings:float=Field(description="The movies ratings out of 10")
    

In [26]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, client=<groq.resources.chat.completions.Completions object at 0x000001B990A4BAD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B990C904D0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'This is the director of the movie', 'type': 'string'}, 'ratings': {'description': 'The movies ratings out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'ratings'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'nam

In [29]:
model.invoke("Provide me the details about the movie Inception")

AIMessage(content='Here are the detailed facts about the movie **Inception**:\n\n### 🎬 Basic Information\n- **Title**: Inception\n- **Release Date**: July 16, 2010 (USA)\n- **Director**: Christopher Nolan\n- **Screenplay**: Christopher Nolan\n- **Producers**: Emma Thomas, Christopher Nolan, Jenna Rose\n- **Music**: Hans Zimmer\n- **Runtime**: 148 minutes\n- **Genre**: Sci-Fi, Action, Thriller, Drama\n- **Studio**: Legendary Pictures, Syncopy, Warner Bros. Pictures\n- **Language**: English\n\n### 🎭 Main Cast\n- **Leonardo DiCaprio** as Dom Cobb\n- **Joseph Gordon-Levitt** as Arthur\n- **Elliot Page** (credited as Ellen Page at the time) as Ariadne\n- **Tom Hardy** as Eames\n- **Ken Watanabe** as Saito\n- **Cillian Murphy** as Robert Fischer\n- **Marion Cotillard** as Mal Cobb\n- **Michael Caine** as Miles\n- **Tom Berenger** as Browning\n\n### 📖 Plot Summary\nDom Cobb (Leonardo DiCaprio) is a skilled thief with a specialized expertise in the art of extraction: the implantation and theft

In [31]:
response = model_with_structure.invoke("Provide details about movie Inception ")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', ratings=8.8)

### Message Output along with Parsed Structure


In [32]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(...,description="The title of movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="This is the director of the movie")
    ratings:float=Field(...,description="The movies ratings out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("Provide details about Prometheus")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'xs7c6ts1q', 'function': {'arguments': '{"director":"Ridley Scott","ratings":6.5,"title":"Prometheus","year":2012}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 352, 'total_tokens': 422, 'completion_time': 0.183772002, 'completion_tokens_details': None, 'prompt_time': 0.02810465, 'prompt_tokens_details': None, 'queue_time': 0.067628169, 'total_time': 0.211876652}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f797-5c4d-7f82-acd9-19c5d8887cca-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Ridley Scott', 'ratings': 6.5, 'title': 'Prometheus', 'year': 2012}, 'id': 'xs7c6ts1q', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 352, 'output_tokens': 70, 'total_tokens

### Nested Structure

In [33]:
from pydantic import BaseModel , Field

class Actor(BaseModel):
    name:str
    age:int
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    director:str
    ratings:float
    budget:float | None = Field(None , description="Budget in Rupees")

model_with_structure=model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about Interstellar")
response

MovieDetails(title='Interstellar', year=2014, cast=[Actor(name='Matthew McConaughey', age=54, role='Cooper'), Actor(name='Anne Hathaway', age=28, role='Brand'), Actor(name='Michael Caine', age=83, role='Professor Brand'), Actor(name='Jessica Chastain', age=36, role='Dr. Mann'), Actor(name='Matt Damon', age=34, role='Doyle'), Actor(name='Casey Affleck', age=30, role='Tommy Cooper'), Actor(name='Mackenzie Foy', age=14, role='Murph Cooper'), Actor(name='Timothée Chalamet', age=49, role='Adult Murph Cooper')], genres=['Adventure', 'Drama', 'Science Fiction'], director='Christopher Nolan', ratings=8.7, budget=7000000000.0)